# Brain MRI Deepfake Generation — GAN Notebook

This notebook follows your folder layout:

`GanBasedSynthesis/<dataset_name>/...`

It supports class-folder datasets, optional DICOM images, and optional BRISC-style masks when present.
Run one dataset at a time by changing `ACTIVE_DATASET_KEY`.

In [1]:
!pip -q install pydicom






In [2]:
!pip install kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 4.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 14.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install torch torchvision --index-url


Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

--index-url option requires 1 argument


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tombackert/brain-tumor-mri-data")

print("Path to dataset files:", path)

/home/rehan/DeepFakes/gpu_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 151M/151M [00:09<00:00, 17.0MB/s] 

Extracting files...


Path to dataset files: /home/rehan/.cache/kagglehub/datasets/tombackert/brain-tumor-mri-data/versions/1


In [6]:
import torch

# Define device: use GPU if available, else fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Example: Move a tensor to the detected device
data = torch.randn(3, 3).to(device)
print(data)

Running on: cuda
tensor([[-0.5950,  0.3828,  1.3523],
        [-0.8066, -1.8687,  0.8421],
        [-0.9509, -0.4446,  1.0012]], device='cuda:0')


In [7]:
import torch
import torchvision

print(torch.__version__)  # Should print the PyTorch version, e.g., 2.0.1
print(torchvision.__version__)  # Should print the torchvision version, e.g., 0.15.0

2.5.1+cu121
0.20.1+cu121


In [8]:
import os, math, random, warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils as vutils

try:
    import pydicom
except Exception:
    pydicom = None

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

ModuleNotFoundError: No module named 'pandas'

In [ ]:
ACTIVE_DATASET_KEY = "brain-tumor-mri-dataset"    # briscdataset  masoudnickparvar	tombackert  unidpro
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_EPOCHS = 50
LR = 2e-4
BETAS = (0.5, 0.999)
Z_DIM = 128
NUM_WORKERS = 2
SAVE_EVERY = 5

CANDIDATE_BASES = [
    Path("./working"),
    Path("./Datasets"),
    Path("."),
]

DATASET_FOLDER_MAP = {
    "unidpro": "unidpro",
    "tombackert": "tombackert",
    "masoudnickparvar": "masoudnickparvar",
    "briscdataset": "briscdataset",
    "mri-dataset":"mri-dataset",
    "brain-tumor-mri-dataset":"brain-tumor-mri-dataset",
}

OUTPUT_ROOT = Path("./working") / "GANBasedSynthesisOutputs"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
SAMPLE_ROOT = OUTPUT_ROOT / "samples"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp", ".dcm"}

LABEL_SYNONYMS = {
    "glioma": "glioma",
    "meningioma": "meningioma",
    "pituitary": "pituitary",
    "pituitary_tumor": "pituitary",
    "pituitarytumor": "pituitary",
    "no_tumor": "notumor",
    "notumor": "notumor",
    "no tumor": "notumor",
    "normal": "notumor",
    "healthy": "notumor",
    "brain_glioma": "glioma",
    "brain_menin": "meningioma",
    "brain tumor": "tumor",
    "tumor": "tumor",
    "all": "all",
}

def normalize_label(name: str) -> str:
    s = name.strip().lower().replace("-", "_").replace(" ", "_")
    return LABEL_SYNONYMS.get(s, s)

def resolve_dataset_root(dataset_key: str) -> Path:
    rel = DATASET_FOLDER_MAP[dataset_key]
    candidates = []
    for base in CANDIDATE_BASES:
        candidates += [
            base / "GanBasedSynthesis" / rel,
            base / "GANBasedSynthesis" / rel,
            base / rel,
            base / dataset_key,
        ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

DATA_ROOT = resolve_dataset_root(ACTIVE_DATASET_KEY)
print("Dataset root:", DATA_ROOT)

In [ ]:
def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMG_EXTS

def read_image(path: Path) -> Image.Image:
    if path.suffix.lower() == ".dcm":
        if pydicom is None:
            raise RuntimeError("pydicom is required for DICOM files.")
        ds = pydicom.dcmread(str(path))
        arr = ds.pixel_array.astype(np.float32)
        arr -= arr.min()
        arr /= (arr.max() - arr.min() + 1e-8)
        return Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    return Image.open(path).convert("RGB")

def read_mask(path: Path) -> Image.Image:
    return Image.open(path).convert("L")

def find_mask_for_image(img_path: Path) -> Optional[Path]:
    stem = img_path.stem
    parent = img_path.parent
    search_dirs = [
        parent,
        parent.parent / "masks",
        parent.parent / "mask",
        parent.parent / "masks_png",
        parent.parent / "labels",
        parent.parent / "seg",
        parent.parent / "segmentation",
    ]
    for d in search_dirs:
        if d.exists():
            for ext in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
                cand = d / f"{stem}{ext}"
                if cand.exists():
                    return cand
    for cand in img_path.parent.rglob(f"{stem}*"):
        if cand.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"} and "mask" in cand.name.lower():
            return cand
    return None

def discover_from_manifest(root: Path):
    csvs = list(root.rglob("manifest.csv"))
    if not csvs:
        return None
    df = pd.read_csv(csvs[0])
    cols = {c.lower(): c for c in df.columns}
    path_col = None
    for c in ["relative_path", "filepath", "image_path", "path"]:
        if c in cols:
            path_col = cols[c]
            break
    label_col = None
    for c in ["tumor_label", "label", "class", "category"]:
        if c in cols:
            label_col = cols[c]
            break
    mask_col = None
    for c in ["mask_path", "relative_mask_path", "mask", "mask_file"]:
        if c in cols:
            mask_col = cols[c]
            break
    if path_col is None:
        return None
    items = []
    for _, row in df.iterrows():
        p = Path(str(row[path_col]))
        img_path = p if p.is_absolute() else root / p
        if not img_path.exists() and "filename" in cols and "relative_path" in cols:
            img_path = root / str(row[cols["relative_path"]]) / str(row[cols["filename"]])
        if not img_path.exists():
            continue
        label = normalize_label(str(row[label_col])) if label_col is not None else "all"
        mask_path = None
        if mask_col is not None:
            mp = Path(str(row[mask_col]))
            mask_path = mp if mp.is_absolute() else root / mp
            if not mask_path.exists():
                mask_path = None
        else:
            mask_path = find_mask_for_image(img_path)
        items.append((img_path, label, mask_path))
    return items if items else None

def discover_items(root: Path):
    manifest_items = discover_from_manifest(root)
    if manifest_items is not None:
        return manifest_items
    items = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_image_file(path):
            continue
        if any(part.lower() in {"mask", "masks", "labels", "seg", "segmentation"} for part in path.parts):
            continue
        label = None
        for part in reversed(path.parts):
            part_n = normalize_label(part)
            if part_n in {"glioma", "meningioma", "pituitary", "notumor", "tumor"}:
                label = part_n
                break
        if label is None:
            label = "all"
        mask_path = find_mask_for_image(path)
        items.append((path, label, mask_path))
    return items

all_items = discover_items(DATA_ROOT)
print("Found items:", len(all_items))
print("Unique labels:", sorted(set(lbl for _, lbl, _ in all_items))[:20])

In [ ]:
from sklearn.model_selection import train_test_split

labels = sorted(set(lbl for _, lbl, _ in all_items))
if len(labels) == 0:
    labels = ["all"]
label2idx = {lbl: i for i, lbl in enumerate(labels)}
idx2label = {i: lbl for lbl, i in label2idx.items()}
NUM_CLASSES = len(labels)
print("NUM_CLASSES =", NUM_CLASSES, label2idx)

train_items, val_items = train_test_split(
    all_items,
    test_size=0.15,
    random_state=42,
    stratify=[x[1] for x in all_items] if len(labels) > 1 else None,
)
print(len(train_items), len(val_items))

In [ ]:
class MRISynthesisDataset(Dataset):
    def __init__(self, items, label2idx, image_size=128, augment=False):
        self.items = items
        self.label2idx = label2idx
        self.augment = augment
        self.size = image_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, label, mask_path = self.items[idx]
        img = read_image(img_path)
        if self.augment:
            img = self.aug(img)
        img = self.to_tensor(img)
        label_idx = self.label2idx[label]
        mask = torch.zeros(1, self.size, self.size)
        has_mask = 0
        if mask_path is not None and mask_path.exists():
            m = read_mask(mask_path).resize((self.size, self.size))
            m = transforms.ToTensor()(m)
            mask = (m > 0.5).float()
            has_mask = 1
        return {
            "image": img,
            "label": torch.tensor(label_idx, dtype=torch.long),
            "mask": mask,
            "has_mask": torch.tensor(has_mask, dtype=torch.long),
            "path": str(img_path),
        }

train_ds = MRISynthesisDataset(train_items, label2idx, IMAGE_SIZE, augment=True)
val_ds = MRISynthesisDataset(val_items, label2idx, IMAGE_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

batch = next(iter(train_loader))
print(batch["image"].shape, batch["label"].shape)

In [ ]:
def denorm(x):
    return (x.clamp(-1, 1) + 1) / 2

def show_batch(batch, n=8, title="sample"):
    imgs = denorm(batch["image"][:n]).cpu()
    grid = vutils.make_grid(imgs, nrow=min(n, 4))
    plt.figure(figsize=(10, 10))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis("off")
    plt.title(title)
    plt.show()

show_batch(batch, n=min(8, len(batch["image"])), title=f"{ACTIVE_DATASET_KEY} samples")

In [ ]:
class GenBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=4, stride=2, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel, stride, padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
        )
    def forward(self, x):
        return self.block(x)

class ConditionalGenerator(nn.Module):
    def __init__(self, z_dim, n_classes, img_channels=3, base_ch=64):
        super().__init__()
        self.z_dim = z_dim
        self.label_emb = nn.Embedding(n_classes, z_dim)
        self.fc = nn.Sequential(
            nn.Linear(z_dim * 2, base_ch * 16 * 4 * 4),
            nn.BatchNorm1d(base_ch * 16 * 4 * 4),
            nn.ReLU(True),
        )
        self.net = nn.Sequential(
            GenBlock(base_ch * 16, base_ch * 8),
            GenBlock(base_ch * 8, base_ch * 4),
            GenBlock(base_ch * 4, base_ch * 2),
            GenBlock(base_ch * 2, base_ch),
            nn.ConvTranspose2d(base_ch, img_channels, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z, y):
        yemb = self.label_emb(y)
        x = torch.cat([z, yemb], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), -1, 4, 4)
        return self.net(x)

class DiscBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class ConditionalDiscriminator(nn.Module):
    def __init__(self, n_classes, img_channels=3, base_ch=64):
        super().__init__()
        self.features = nn.Sequential(
            DiscBlock(img_channels, base_ch, norm=False),
            DiscBlock(base_ch, base_ch * 2),
            DiscBlock(base_ch * 2, base_ch * 4),
            DiscBlock(base_ch * 4, base_ch * 8),
            DiscBlock(base_ch * 8, base_ch * 16),
        )
        feat_dim = base_ch * 16 * 4 * 4
        self.real_head = nn.Linear(feat_dim, 1)
        self.class_head = nn.Linear(feat_dim, n_classes)

    def forward(self, x):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        return self.real_head(f).squeeze(1), self.class_head(f)

G = ConditionalGenerator(Z_DIM, NUM_CLASSES).to(DEVICE)
D = ConditionalDiscriminator(NUM_CLASSES).to(DEVICE)
print(sum(p.numel() for p in G.parameters()) / 1e6, "M params G")
print(sum(p.numel() for p in D.parameters()) / 1e6, "M params D")

In [ ]:
bce = nn.BCEWithLogitsLoss()
ce = nn.CrossEntropyLoss()

opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)

fixed_z = torch.randn(NUM_CLASSES * 8, Z_DIM, device=DEVICE)
fixed_y = torch.tensor([i for i in range(NUM_CLASSES) for _ in range(8)], dtype=torch.long, device=DEVICE)

def save_grid_from_generator(step_name: str):
    G.eval()
    with torch.no_grad():
        fake = G(fixed_z, fixed_y).cpu()
    fake = denorm(fake)
    grid = vutils.make_grid(fake, nrow=8)
    out = SAMPLE_ROOT / f"{ACTIVE_DATASET_KEY}_{step_name}.png"
    plt.figure(figsize=(12, 12))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", out)
    G.train()

def train_one_epoch():
    G.train()
    D.train()
    g_loss_meter = 0.0
    d_loss_meter = 0.0
    for batch in train_loader:
        real = batch["image"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        bs = real.size(0)

        z = torch.randn(bs, Z_DIM, device=DEVICE)
        fake = G(z, y).detach()

        real_logit, real_cls = D(real)
        fake_logit, fake_cls = D(fake)

        valid = torch.ones_like(real_logit)
        fake_target = torch.zeros_like(fake_logit)

        d_loss = (
            bce(real_logit, valid) +
            bce(fake_logit, fake_target) +
            ce(real_cls, y) +
            ce(fake_cls, y)
        )

        opt_d.zero_grad(set_to_none=True)
        d_loss.backward()
        opt_d.step()

        z = torch.randn(bs, Z_DIM, device=DEVICE)
        gen = G(z, y)
        gen_logit, gen_cls = D(gen)
        g_loss = bce(gen_logit, valid) + ce(gen_cls, y)

        opt_g.zero_grad(set_to_none=True)
        g_loss.backward()
        opt_g.step()

        g_loss_meter += g_loss.item()
        d_loss_meter += d_loss.item()

    return g_loss_meter / max(1, len(train_loader)), d_loss_meter / max(1, len(train_loader))

def eval_preview():
    G.eval()
    with torch.no_grad():
        batch = next(iter(val_loader))
        real = batch["image"][:8].to(DEVICE)
        y = batch["label"][:8].to(DEVICE)
        z = torch.randn(real.size(0), Z_DIM, device=DEVICE)
        fake = G(z, y).cpu()
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for i in range(8):
        axes[0, i].imshow(np.transpose(denorm(real.cpu())[i].numpy(), (1, 2, 0)))
        axes[0, i].axis("off")
        axes[1, i].imshow(np.transpose(denorm(fake)[i].numpy(), (1, 2, 0)))
        axes[1, i].axis("off")
    axes[0, 0].set_title("real")
    axes[1, 0].set_title("fake")
    plt.tight_layout()
    plt.show()

best_loss = float("inf")
for epoch in range(1, NUM_EPOCHS + 1):
    g_loss, d_loss = train_one_epoch()
    print(f"Epoch {epoch:03d} | G: {g_loss:.4f} | D: {d_loss:.4f}")
    if epoch % SAVE_EVERY == 0:
        save_grid_from_generator(f"epoch_{epoch:03d}")
        ckpt = {
            "epoch": epoch,
            "G": G.state_dict(),
            "D": D.state_dict(),
            "label2idx": label2idx,
            "dataset_key": ACTIVE_DATASET_KEY,
        }
        torch.save(ckpt, CHECKPOINT_ROOT / f"{ACTIVE_DATASET_KEY}_gan_epoch_{epoch:03d}.pt")
    if d_loss < best_loss:
        best_loss = d_loss
        torch.save(G.state_dict(), CHECKPOINT_ROOT / f"{ACTIVE_DATASET_KEY}_gan_bestG.pt")

eval_preview()

In [ ]:
# Optional export: generate a synthetic set from the trained generator
G.eval()

GEN_OUT = OUTPUT_ROOT / ACTIVE_DATASET_KEY / "generated"
GEN_OUT.mkdir(parents=True, exist_ok=True)

N_PER_CLASS = 1500   # match your previous N_GEN
BATCH = 32           # keep GPU-safe batching

with torch.no_grad():
    for cls_idx in range(NUM_CLASSES):

        cls_name = idx2label.get(cls_idx, f"class_{cls_idx}")
        class_dir = GEN_OUT / cls_name
        class_dir.mkdir(exist_ok=True)

        img_counter = 0

        for i in range(0, N_PER_CLASS, BATCH):

            current_batch = min(BATCH, N_PER_CLASS - i)

            # GAN inputs
            z = torch.randn(current_batch, Z_DIM, device=DEVICE)
            y = torch.full((current_batch,), cls_idx, dtype=torch.long, device=DEVICE)

            fake = G(z, y).cpu()

            for j in range(fake.size(0)):
                img = denorm(fake[j]).clamp(0, 1)
                arr = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

                Image.fromarray(arr).save(
                    class_dir / f"{cls_name}_{img_counter:05d}.png"
                )

                img_counter += 1

print("Saved synthetic images to:", GEN_OUT)

In [ ]:

# ===== ZIP EVERYTHING =====
zip_path = OUTPUT_ROOT / f"{ACTIVE_DATASET_KEY}_GAN_outputs.zip"

shutil.make_archive(
    str(zip_path).replace(".zip", ""),  # remove .zip extension
    'zip',
    root_dir=OUT_DIR
)

print("📦 ZIP saved at:", zip_path)

# ===== DOWNLOAD LINK =====
from IPython.display import FileLink
display(FileLink(zip_path))

## Notes for BRISC-style masks

If the dataset contains segmentation masks, the notebook already keeps them in the sample dictionary.
That lets you build mask-guided deepfake variants later by blending or by training a conditional editing model.

For the first synthesis pass, this notebook generates class-consistent fake MRI slices.